# Task 1: Approximating Continuous Functions with Neural Networks

In this notebook we train a neural network to approximate a continuous mathematical function from sampled data.

The idea is the following:
1. generate input values;
2. compute the corresponding outputs using a known function;
3. train the neural network to learn the relationship between inputs and outputs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import random
import torch
import torch.optim
from torch import nn

from sklearn.preprocessing import StandardScaler

## Creating the Neural Network

The function below creates a feedforward neural network with fully connected (`Linear`) layers and `ReLU` activation functions.

The list `layers` specifies the size of each layer of the network.


In [ ]:
def create_network(layers):
    return nn.Sequential(*[
        module
        for x, y in zip(layers, layers[1:])
        for module in [nn.Linear(x, y), nn.ReLU()]
    ][:-1])

## Generating the Dataset

We generate random input values and compute the corresponding outputs using a known mathematical function.

In this first example, the target function is

$$y = 5x.$$

Then the network will try to learn this relationship directly from the data.


## Standardizing the Inputs

The inputs are standardized using `StandardScaler`.

This transformation rescales the data so that it has:
- mean approximately equal to 0;
- standard deviation approximately equal to 1.

Normalization often improves the stability and speed of training.


In [ ]:
# Create training dataset
N = 100_000
raw_inputs  = (np.random.rand(N,1) - 1/2) * 100
raw_outputs = 5*raw_inputs

# Normalizing the data
input_scaler = StandardScaler()
output_scaler = StandardScaler()

transformed_inputs = input_scaler.fit_transform(raw_inputs)
transformed_outputs = output_scaler.fit_transform(raw_outputs)

inputs = torch.tensor(transformed_inputs, dtype=torch.float32)
outputs = torch.tensor(transformed_outputs, dtype=torch.float32)

## Defining the Model

We now create the neural network model, choose a loss function, and define the optimizer.

The optimizer updates the parameters of the network during training in order to minimize the loss function.

In [ ]:
TRAINING_FACTOR = 2

indices = torch.randperm(N)
training_idx, validation_idx = indices[:N // TRAINING_FACTOR], indices[N // TRAINING_FACTOR:]

input_training, output_training = inputs[training_idx], outputs[training_idx]
input_validation, output_validation = inputs[validation_idx], outputs[validation_idx]

def create_and_train_model(layers, epochs):
    model = create_network(layers)
    loss_function = nn.MSELoss()
    optimiser = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.01)

    # We will record the loss functions across the dataset every LOSS_INTERVAL epochs.
    LOSS_INTERVAL = 1000
    BATCH_SIZE = 100
    recorded_loss = torch.full(size=(epochs//LOSS_INTERVAL, 3), fill_value=0.0)

    # At each epoch, show the network BATCH_SIZE data points and do gradient descent.
    for epoch in range(epochs):
        training_size = input_training.shape[0]
        batch = torch.tensor(random.sample(range(training_size), k=BATCH_SIZE))

        optimiser.zero_grad()

        model_output = model(input_training[batch])

        loss = loss_function(model_output, output_training[batch])
        loss.backward()
        optimiser.step()

        if epoch % LOSS_INTERVAL == 0:
            # Record the training and validation loss of the network at each epoch.
            # We do this with no_grad enabled: this disables computation tracking and uses much less RAM.
            print(epoch)
            with torch.no_grad():
                recorded_loss[epoch // LOSS_INTERVAL, 0] = epoch
                recorded_loss[epoch // LOSS_INTERVAL, 1] = loss_function(model(input_training), output_training)
                recorded_loss[epoch // LOSS_INTERVAL, 2] = loss_function(model(input_validation), output_validation)

    return model, recorded_loss

## Training the Network

During training:
1. the model computes predictions;
2. the loss function measures the error;
3. backpropagation computes the gradients;
4. the optimizer updates the parameters.

The loss should decrease as training progresses.


In [ ]:
model, loss = create_and_train_model([1, 50, 50, 50, 50, 1], epochs=5_000)

# Plot the loss function over time.
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(loss[:, 0], loss[:, 1], label='Training loss')
ax.plot(loss[:, 0], loss[:, 2], label='Validation loss')
ax.set_xlabel('Epoch')
ax.legend()
ax.grid()
ax.set_title("Loss over time")
plt.show()

## Visualizing the Results

Finally, we compare the predictions of the neural network with the true values of the target function.

If the training was successful, the predicted curve should closely match the original function.


In [ ]:
# Plot the loss function over time.
fig, ax = plt.subplots(figsize=(20, 8))

ax.scatter(inputs, outputs, label='sin')
y = model(inputs)
y = y.detach().numpy()
# print(y[:10])
ax.scatter(inputs, y)

# ax.scatter(inputs, model(inputs)[:,0], label='?')
ax.grid()
#plt.show()

# Task 2: Approximation of the sin
The goal of this task is to approximate the sin function using the privious example as guide.

Can the model predict the sin outside of the boundaries of the dataset? 
What if we do not normalize the data? 